In [1]:
import os
import re
import numpy as np
import pandas as pd
import nibabel as nib

In [23]:
# End goal --> create an annotation for each volume
# 746 volumes
746 * 0.8 # 596 seconds

596.8000000000001

In [50]:
# keystrokes
def print_answer(timestamps, answer):
    diff = (timestamps[-1] - timestamps[0])/10**3 # duration based on keystrokes
    print('\nnew stimulus') 
    print(diff) 
    print([(float(t)/10**3) - magic_number for t in timestamps])
    print(answer) 
    return diff

with open('../data/125/keystrokes-125-3.txt', 'r') as f:
    answer = '' # where the participant's response will be accumulated
    timestamps = [] # all the timestamps for each keystroke for a participant response
    durations = [] # all the durations based on the keystroke data
    keystrokes_df = []
    
    lines = f.readlines()
    for i, line in enumerate(lines):
        asci = re.split(',', line.strip())
        keystrokes_df.append(asci)
        if (len(asci) <= 1 and i != 0) or (i == len(lines) - 1): # if it's 'new stimulus' or '<timestamp>, <ascii key>'
            diff = print_answer(timestamps, answer)              #      or the first or last stimulus
            answer = ''
            durations.append(diff)
            timestamps = []
        elif len(asci) == 2: # if it's the comma separated timestamp and keystroke
            asci_int = int(asci[1])
            asci_chr = chr(asci_int)
            ts = float(asci[0])
            timestamps.append(ts)
            answer += asci_chr
            
keystrokes_df = pd.DataFrame(keystrokes_df)
keystrokes_df.columns = ['timestamps', 'ascii_code']
# keystrokes_df = keystrokes_df.loc[1:]


new stimulus
45.840530615091325
[30.145094966283068, 32.65647555026226, 33.038857963401824, 33.1882109651342, 33.6151444693096, 33.77296071033925, 34.00340179610066, 34.5751155950129, 35.03706194832921, 35.25547800422646, 35.62092117709108, 35.782724771182984, 36.004155268194154, 36.610500464215875, 36.787059729220346, 38.621808087220415, 38.62268556514755, 38.622922759270296, 39.39326233230531, 39.393597178161144, 39.3938365902286, 39.39417254622094, 39.52852824423462, 39.57546236924827, 39.76249921717681, 39.96301477425732, 41.99527451233007, 42.306087107164785, 42.31633249018341, 42.89781256811693, 43.08840787713416, 43.33906980021857, 43.45944993221201, 43.58603483019397, 43.731457074172795, 44.92171781510115, 45.65042419615202, 46.16547809308395, 46.196069152094424, 46.22518739127554, 46.26667854306288, 46.287639814196154, 46.318960764212534, 46.349817036185414, 46.381864270195365, 46.41839516116306, 47.313660939224064, 47.89266588026658, 48.490211327327415, 49.18257019203156, 49

In [51]:
new_stim_idx = np.where(keystrokes_df['timestamps'] == 'new stimulus')[0]
keystrokes_df.drop(new_stim_idx, inplace=True)
keystrokes_df = keystrokes_df.reset_index()

keystrokes_df['timestamps'] = keystrokes_df['timestamps'].apply(lambda x: (float(x)/10**3) - magic_number) 
keystrokes_df['ascii_code'] = keystrokes_df['ascii_code'].apply(lambda x: chr(int(x)))

In [3]:
df = []

with open('../data/125/processed-answers-125-3.txt', 'r') as f:
    for line in f:
        # print(line.strip())
        newline = line.strip()
        df.append(re.split(',', newline))
df = pd.DataFrame(df)
df.columns = df.iloc[0]
df = df[1:].reset_index()

In [4]:
df['timestamp'].apply(lambda x: float(x)/10**3)

0    1.112148e+06
1    1.112211e+06
2    1.112277e+06
3    1.112345e+06
4    1.112410e+06
5    1.112477e+06
6    1.112543e+06
7    1.112607e+06
8    1.112670e+06
Name: timestamp, dtype: float64

In [5]:
# align first timestamp in processed answers (ms) to second timestamp in relative onsets (s)


onset_df = []
with open('../data/125/relative-onsets-125-3.txt', 'r') as f:
    for line in f:
        newline = line.strip()
        onset_df.append(re.split('\s', newline))
onset_df = pd.DataFrame(onset_df)
onset_df.columns = ['stim_id', 'timestamp']
print(onset_df)

  stim_id      timestamp
0     8.0  13.2814625953
1     4.0  76.2640425031
2     2.0  142.331810221
3     7.0  209.398675885
4     5.0  274.724996479
5     1.0  341.653943194
6     0.0  408.206410653
7     3.0  471.434752137
8     6.0          535.0


In [6]:
ts_to_match = float(onset_df.loc[1, 'timestamp'])
end_ts = float(df.loc[0,'timestamp'])/10**3
magic_number = (ts_to_match - end_ts) * (-1)
df['timestamp'].apply(lambda x: (float(x)/10**3) - magic_number) # this gives end timestamps
# print((ts_to_match - end_ts)*-1)

0     76.264043
1    139.285134
2    205.377625
3    272.471747
4    337.837349
5    404.789678
6    471.367185
7    534.624506
8    598.213071
Name: timestamp, dtype: float64

In [56]:
vol_number = 0
for rt in range(800, 597600, 800):
    window_start = rt/10**3
    window_end = (window_start + 0.8)
    
    # iterate through keystrokes and find the ones within the range
    writing = ''
    for i in range(len(keystrokes_df)):
        key_time = float(keystrokes_df.loc[i, 'timestamps'])
        if window_end < key_time:
            continue
        elif window_end > key_time and window_start < key_time:
            writing += keystrokes_df.loc[i, 'ascii_code'] 
    
    if writing:
        print(f'volume: {vol_number} | window end: {window_start} | window end: {window_end} | writing: {writing}')
        
    vol_number += 1
    
    
    # the time in the volumes between the trial start and the first time typing could also be interesting
    

volume: 36 | window end: 29.6 | window end: 30.400000000000002 | writing: V
volume: 39 | window end: 32.0 | window end: 32.8 | writing: O
volume: 40 | window end: 32.8 | window end: 33.599999999999994 | writing: UD
volume: 41 | window end: 33.6 | window end: 34.4 | writingU
volume: 42 | window end: 34.4 | window end: 35.199999999999996 | writing:O
volume: 43 | window end: 35.2 | window end: 36.0 | writing:
volume: 44 | window end: 36.0 | window end: 36.8 | writing: ID 
volume: 47 | window end: 38.4 | window end: 39.199999999999996 | writing: COU
volume: 48 | window end: 39.2 | window end: 40.0 | writing: NTDOWN9
volume: 51 | window end: 41.6 | window end: 42.4 | writing: ITN
volume: 52 | window end: 42.4 | window end: 43.199999999999996 | writing
volume: 53 | window end: 43.2 | window end: 44.0 | writing: NT N
volume: 55 | window end: 44.8 | window end: 45.599999999999994 | writing: '
volume: 56 | window end: 45.6 | window end: 46.4 | writing: 
volume: 57 | window end: 46.4 |

In [31]:
# initial_time = float(df.loc[0,'timestamp'])
initial_time = 1112102202.303803 # first timestamp from keystrokes
# doesn't make sense to add relative onset since this keystroke time only corresponds to when the participant started typing
#  + 13.2814625953
df['timestamp'].apply(lambda x: ((float(x)/10**3) - (initial_time/10**3)))

0     46.118948
1    109.140039
2    175.232530
3    242.326652
4    307.692254
5    374.644583
6    441.222090
7    504.479411
8    568.067976
Name: timestamp, dtype: float64

In [29]:
volumes = {el: '' for el in range(746)}
volumes

{0: '',
 1: '',
 2: '',
 3: '',
 4: '',
 5: '',
 6: '',
 7: '',
 8: '',
 9: '',
 10: '',
 11: '',
 12: '',
 13: '',
 14: '',
 15: '',
 16: '',
 17: '',
 18: '',
 19: '',
 20: '',
 21: '',
 22: '',
 23: '',
 24: '',
 25: '',
 26: '',
 27: '',
 28: '',
 29: '',
 30: '',
 31: '',
 32: '',
 33: '',
 34: '',
 35: '',
 36: '',
 37: '',
 38: '',
 39: '',
 40: '',
 41: '',
 42: '',
 43: '',
 44: '',
 45: '',
 46: '',
 47: '',
 48: '',
 49: '',
 50: '',
 51: '',
 52: '',
 53: '',
 54: '',
 55: '',
 56: '',
 57: '',
 58: '',
 59: '',
 60: '',
 61: '',
 62: '',
 63: '',
 64: '',
 65: '',
 66: '',
 67: '',
 68: '',
 69: '',
 70: '',
 71: '',
 72: '',
 73: '',
 74: '',
 75: '',
 76: '',
 77: '',
 78: '',
 79: '',
 80: '',
 81: '',
 82: '',
 83: '',
 84: '',
 85: '',
 86: '',
 87: '',
 88: '',
 89: '',
 90: '',
 91: '',
 92: '',
 93: '',
 94: '',
 95: '',
 96: '',
 97: '',
 98: '',
 99: '',
 100: '',
 101: '',
 102: '',
 103: '',
 104: '',
 105: '',
 106: '',
 107: '',
 108: '',
 109: '',
 110: '',


In [43]:
with open('../data/125/relative-onsets-125-3.txt', 'r') as f:
    test = f.read()
print(test)

8.0 13.2814625953
4.0 76.2640425031
2.0 142.331810221
7.0 209.398675885
5.0 274.724996479
1.0 341.653943194
0.0 408.206410653
3.0 471.434752137
6.0 535.0



In [79]:
535/60

8.916666666666666